In [20]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.cm as cm
from matplotlib_venn import venn2
from adjustText import adjust_text
import decoupler as dc
import ast

In [21]:
import session_info2; session_info2.session_info(dependencies=True)

Package,Version
pandas,2.3.2
numpy,2.2.6
matplotlib,3.10.6
matplotlib-venn,1.1.2
seaborn,0.13.2
scipy,1.15.3
adjustText,1.3.0
decoupler,2.1.6
Component,Info
Python,"3.13.7 | packaged by conda-forge | (main, Sep 3 2025, 14:30:35) [GCC 14.3.0]"


# Panel E

In [ ]:
# Read DGE results
df_rcvb6 = pd.read_csv('tables/RCvB6_DGE.csv', index_col=0)
df_dfmovrc = pd.read_csv('tables/DFMOvRC_DGE.csv', index_col=0)
tables/ B6
up_in_rc = df_rcvb6[
    (df_rcvb6['log2FoldChange'] > 1) &
    (df_rcvb6['padj'] < 0.05)
]

# Genes suppressed in RC vs B6
down_in_rc = df_rcvb6[
    (df_rcvb6['log2FoldChange'] < -1) &
    (df_rcvb6['padj'] < 0.05)
]

# Genes upregulated in DFMO vs RC
up_in_dfmo = df_dfmovrc[
    (df_dfmovrc['log2FoldChange'] > 1) &
    (df_dfmovrc['padj'] < 0.05)
]

# Genes suppressed in DFMO vs RC
down_in_dfmo = df_dfmovrc[
    (df_dfmovrc['log2FoldChange'] < -1) &
    (df_dfmovrc['padj'] < 0.05)
]

In [ ]:
#plotting venn

# Convert to sets
set_up_in_rc = set(up_in_rc.index)
set_down_in_dfmo = set(down_in_dfmo.index)

set_down_in_rc = set(down_in_rc.index)
set_up_in_dfmo = set(up_in_dfmo.index)

fig, axes = plt.subplots(2, 1, figsize=(5, 8))

# Upregulated genes rescued
plt.sca(axes[0])  # set current axis
venn2(
    [set_up_in_rc, set_down_in_dfmo],
    set_labels=('Up in RC', 'Down in DFMO'),
    set_colors=('tomato', 'steelblue'),
    alpha=0.6,
    ax=axes[0],
)
axes[0].set_title("Upregulated Genes Rescued", fontsize=13, fontweight='bold')

# Downregulated genes rescued
plt.sca(axes[1])  # set current axis
venn2(
    [set_down_in_rc, set_up_in_dfmo],
    set_labels=('Down in RC', 'Up in DFMO'),
    set_colors=('tomato', 'steelblue'),
    alpha=0.6,
    ax=axes[1],
)
axes[1].set_title("Downregulated Genes Rescued", fontsize=13, fontweight='bold')

#fig.suptitle("Rescued Genes by DFMO Treatment", fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
#plt.savefig('pdfs/panel_e.pdf', dpi=300, bbox_inches='tight')
plt.show()

# Panel F

In [ ]:
# Read pathway results and filter by pvalue
pw_rcvb6 = pd.read_csv('tables/RCvB6_DGE_PW.csv', index_col=0)
pw_rcvb6 = pw_rcvb6[pw_rcvb6['pvalue'] < 0.05]
pw_dfmovrc = pd.read_csv('tables/DFMOvRC_DGE_PW.csv', index_col=0)
pw_dfmovrc = pw_dfmovrc[pw_dfmovrc['pvalue'] < 0.05]

#find rescue genes
rescue_genelist = list(set_up_in_rc.intersection(set_down_in_dfmo)) + list(set_down_in_rc.intersection(set_up_in_dfmo))
rescue_set = set(rescue_genelist)

# filter for pathways containing genes in rescue_genelist

def parse_gene_list(val):
    if isinstance(val, list):
        return val
    try:
        return ast.literal_eval(val)   # handles "['Abcc2', 'Abcc6']" strings
    except (ValueError, SyntaxError):
        return []

mask = pw_rcvb6['features with 2 FC'].apply(
    lambda val: bool(rescue_set.intersection(parse_gene_list(val)))
)

pw_rcvb6_filtered = pw_rcvb6[mask]

mask = pw_dfmovrc['features with 2 FC'].apply(
    lambda val: bool(rescue_set.intersection(parse_gene_list(val)))
)

pw_dfmovrc_filtered = pw_dfmovrc[mask]

In [ ]:
# Top 15 highest ranked in RC_vs_B6
top_rc_up = pw_rcvb6_filtered[pw_rcvb6_filtered['score'] > 0].nlargest(15, 'score').index

# Top 15 lowest (most negative) ranked in DFMO_vs_RC
top_dfmo_down = pw_dfmovrc_filtered[pw_dfmovrc_filtered['score'] < 0].nsmallest(15, 'score').index

# Overlap
overlap_up = top_rc_up.intersection(top_dfmo_down)
print(len(overlap_up))

In [ ]:
# Top 15 lowest in RC_vs_B6
top_rc_down = pw_rcvb6_filtered[pw_rcvb6_filtered['score'] < 0].nsmallest(15, 'score').index

# Top 10 lowest (most negative) in DFMO_vs_RC
top_dfmo_up = pw_dfmovrc_filtered[pw_dfmovrc_filtered['score'] > 0].nlargest(15, 'score').index

# Overlap
overlap_down = top_rc_down.intersection(top_dfmo_up)
print(len(overlap_down))

In [ ]:
pw_rcvb6_filter_plot = pw_rcvb6_filtered.loc[list(overlap_up)+list(overlap_down)]
pw_rcvb6_filter_plot['variable'] = pw_rcvb6_filter_plot.index
dc.pl.dotplot(df=pw_rcvb6_filter_plot, x="score", y="variable", s="logpval", c="score", scale=0.5, figsize=(5.5, 3.5), top=15, vcenter = 0)
plt.title('RC vs B6', fontsize=12, pad=10)
plt.tight_layout()
#plt.savefig(f"pdfs/pathways_rcvb6_overlap.pdf", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
pw_dfmovrc_filter_plot = pw_dfmovrc_filtered.loc[list(overlap_up)+list(overlap_down)]
pw_dfmovrc_filter_plot['variable'] = pw_dfmovrc_filter_plot.index
dc.pl.dotplot(df=pw_dfmovrc_filter_plot, x="score", y="variable", s="logpval", c="score", scale=0.5, figsize=(5.5, 3.5), top=15, vcenter = 0)
plt.title('DMFO vs RC', fontsize=12, pad=10)
plt.tight_layout()
#plt.savefig(f"pdfs/pathways_dmfovrc_overlap.pdf", dpi=300, bbox_inches="tight")
plt.show()

# Panel G CIBERSORTx

In [ ]:
# Read data and calculate average across the samples
cibrsort_df = pd.read_csv('tables/CIBERSORTx_Job4_Results.csv', index_col=0)
cibrsort_df_plot = cibrsort_df.iloc[:, :-3]
# Average every 4 rows (3 groups of 4) the samples are in order
cibersort_avg = (cibrsort_df_plot
                 .reset_index(drop=True)
                 .groupby(lambda x: x // 4)
                 .mean())

cibersort_avg.index = ['B6', 'RC', 'DFMO']

In [ ]:
# Define ordered columns
col_order = [
    'PTS1', 'PTS2', 'PTS3', 'FRPTC_PEC',
    'DTL1', 'DTL2', 'ATL', 'TAL',
    'DCT', 'CNT', 'PC1', 'PC2',
    'URO', 'ICA', 'ICB', 'PODO',
    'ENDO', 'FIB', 
    'Myel', 'Tcell', 'Bcell', 'FAT'
]

# Reorder columns
df_plot = cibersort_avg[col_order]

# Generate colors
colors = (
    [cm.tab20(i) for i in range(20)] +
    [cm.tab20b(i) for i in range(2)]   # take 2 extra from tab20b
)
# Plot
fig, ax = plt.subplots(figsize=(4, 5))

# Reverse order so PTS1 starts from top
col_order_rev = col_order[::-1]
colors_rev = colors[::-1]

bottom = np.zeros(len(df_plot))
for i, col in enumerate(col_order_rev):
    ax.bar(df_plot.index, df_plot[col], bottom=bottom, color=colors_rev[i], label=col)
    bottom += df_plot[col].values

# Formatting
ax.set_ylabel('Cell Type Proportion', fontsize=12)
ax.set_xlabel('Condition', fontsize=12)
ax.set_title('CIBERSORTx Deconvolution Results', fontsize=14, fontweight='bold')
ax.set_xticklabels(df_plot.index, rotation=45, ha='right')

# Remove grids and spines
ax.grid(False)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Legend (keep original order so legend reads top to bottom)
legend_handles = [
    mpatches.Patch(color=colors[i], label=f"{col}")
    for i, col in enumerate(col_order)
]
ax.legend(
    handles=legend_handles,
    bbox_to_anchor=(1.05, 1),
    loc='upper left',
    fontsize=8,
    frameon=False,
)

plt.tight_layout()
#plt.savefig(f"pdfs/cibersort_conditions.pdf", dpi=300, bbox_inches="tight")
plt.show()